# Extração de Metadados dos Artigos ICCRTS via Groq LLM

Este notebook extrai metadados estruturados (~474 artigos) das edições 21–30 do **International Command and Control Research & Technology Symposium (ICCRTS)**, utilizando o modelo `openai/gpt-oss-120b` via API Groq.

Os arquivos Markdown (convertidos de PDF/DOCX com `docling`) estão armazenados no Google Drive sob:
```
/content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS/
```

**Metadados extraídos por artigo:** TITLE, AUTHORS, AUTHORS_APA, KEYWORDS, ABSTRACT, REFERENCES, CONFERENCE, YEAR, SOURCE_FILE, SOURCE_FOLDER.

In [ ]:
# Cell 2: Install dependencies
!pip install groq pandas openpyxl tqdm

In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4: Configuration constants
from google.colab import userdata

# Groq API key (stored as a Colab Secret)
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Base path on Google Drive
BASE_PATH = '/content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS'

# Model to use
MODEL = 'openai/gpt-oss-120b'

# Target folders (editions 21-30)
TARGET_FOLDERS = [
    'ICCRTS_21_ano_2016_MD',
    'ICCRTS_22_ano_2017_MD',
    'ICCRTS_23_ano_2018_MD',
    'ICCRTS_24_ano_2019_MD',
    'ICCRTS_25_ano_2020_MD',
    'ICCRTS_26_ano_2021_MD',
    'ICCRTS_27_ano_2022_MD',
    'ICCRTS_28_ano_2023_MD',
    'ICCRTS_29_ano_2024_MD',
    'ICCRTS_30_ano_2025_MD',
]

# Checkpoint file path (incremental saves to avoid losing data on session disconnect)
CHECKPOINT_PATH = BASE_PATH + '/checkpoint_metadata.json'

# Maximum characters per article sent to the LLM (~60,000 chars ≈ 15,000 tokens)
MAX_CONTENT_CHARS = 60_000

# Save checkpoint every N processed files
SAVE_EVERY = 5

print(f"Configuration loaded.")
print(f"BASE_PATH : {BASE_PATH}")
print(f"MODEL     : {MODEL}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
# Cell 5: Helper functions — parse_folder_metadata() and build_file_manifest()
import re
from pathlib import Path


def parse_folder_metadata(folder_name: str) -> dict:
    """Extract CONFERENCE and YEAR from folder names matching ICCRTS_{num}_ano_{year}_MD."""
    match = re.match(r'ICCRTS_(\d+)_ano_(\d{4})_MD', folder_name)
    if match:
        num = int(match.group(1))
        year = int(match.group(2))
        # Build ordinal suffix
        if num == 21:
            ordinal = '21st'
        elif num == 22:
            ordinal = '22nd'
        elif num == 23:
            ordinal = '23rd'
        else:
            ordinal = f'{num}th'
        conference = f'{ordinal} International Command and Control Research & Technology Symposium'
        return {'CONFERENCE': conference, 'YEAR': year}
    return {'CONFERENCE': '', 'YEAR': ''}


def build_file_manifest(base_path: str, target_folders: list) -> list:
    """Scan target folders; return list of dicts with filepath, folder_name, filename."""
    manifest = []
    base = Path(base_path)
    for folder in target_folders:
        folder_path = base / folder
        if folder_path.exists():
            for md_file in sorted(folder_path.glob('*.md')):
                manifest.append({
                    'filepath': md_file,
                    'folder_name': folder,
                    'filename': md_file.name,
                })
        else:
            print(f'WARNING: folder not found → {folder_path}')
    return manifest


print('Helper functions defined.')

In [ ]:
# Cell 6: Build and display file manifest
import pandas as pd

manifest = build_file_manifest(BASE_PATH, TARGET_FOLDERS)

print(f'Total markdown files found: {len(manifest)}\n')

# Count per folder
folder_counts = {}
for item in manifest:
    folder_counts[item['folder_name']] = folder_counts.get(item['folder_name'], 0) + 1

print('Files per folder:')
for folder, count in folder_counts.items():
    meta = parse_folder_metadata(folder)
    print(f'  {folder:35s}  {count:3d} files  ({meta["CONFERENCE"]})')

In [ ]:
# Cell 7: LLM prompt templates

SYSTEM_PROMPT = """You are an expert academic metadata extractor. You will receive the full text of a scientific article in Markdown format. Your task is to extract specific metadata fields and return them in a strict JSON format.

RULES:
1. Extract exactly the fields requested.
2. If a field is not found in the document, return an empty string "".
3. For AUTHORS: list all author names separated by semicolons. e.g., "John Smith; Jane Doe; Robert Brown"
4. For AUTHORS_APA: convert each author name to APA format: "Last, F. I." separated by semicolons. e.g., "Smith, J.; Doe, J.; Brown, R."
5. For KEYWORDS: list keywords separated by semicolons. If no keywords section exists, return "".
6. For ABSTRACT: return the full abstract text. If no explicit abstract section, return "".
7. For REFERENCES: return each reference as a complete citation string, separated by the pipe character "|". e.g., "Smith, J. (2020). Title. Journal, 1(2), 3-4.|Doe, J. (2019). Another Title. Conference."
8. Return ONLY valid JSON, no markdown code fences, no explanation.

OUTPUT FORMAT (strict JSON):
{\n  \"TITLE\": \"...\",\n  \"AUTHORS\": \"...\",\n  \"AUTHORS_APA\": \"...\",\n  \"KEYWORDS\": \"...\",\n  \"ABSTRACT\": \"...\",\n  \"REFERENCES\": \"...\"\n}"""


def build_user_prompt(markdown_content: str) -> str:
    return f'Extract metadata from the following scientific article:\n\n---\n{markdown_content}\n---'


print('Prompt templates defined.')
print(f'System prompt length: {len(SYSTEM_PROMPT)} characters.')

In [ ]:
# Cell 8: extract_metadata_from_markdown() — Groq API call with retry logic
import json
import time
from groq import Groq

# Instantiate Groq client
client = Groq(api_key=GROQ_API_KEY)


def extract_metadata_from_markdown(
    client: Groq,
    markdown_content: str,
    model: str,
    max_chars: int,
    max_retries: int = 3,
) -> dict:
    """
    Send markdown content to Groq LLM; parse and return the JSON metadata dict.
    Retries up to max_retries times on transient errors (rate limits, timeouts).
    On unrecoverable errors returns a dict with an _ERROR key.
    """
    # --- Smart truncation: keep 80% head + 20% tail ---
    if len(markdown_content) > max_chars:
        head_size = int(max_chars * 0.8)
        tail_size = max_chars - head_size
        markdown_content = (
            markdown_content[:head_size]
            + '\n\n[...CONTENT TRUNCATED...]\n\n'
            + markdown_content[-tail_size:]
        )

    # Skip empty files
    if not markdown_content.strip():
        return {
            'TITLE': '', 'AUTHORS': '', 'AUTHORS_APA': '',
            'KEYWORDS': '', 'ABSTRACT': '', 'REFERENCES': '',
            '_ERROR': 'Empty markdown file',
        }

    user_prompt = build_user_prompt(markdown_content)

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_prompt},
                ],
                temperature=0.1,
                max_completion_tokens=8192,
                top_p=1,
                stream=False,
            )

            response_text = completion.choices[0].message.content.strip()

            # Strip potential markdown code fences  (```json ... ```)
            if response_text.startswith('```'):
                lines = response_text.split('\n')
                # Remove first line (```json or ```) and last line (```)
                response_text = '\n'.join(lines[1:-1]).strip()

            metadata = json.loads(response_text)
            return metadata  # Success

        except json.JSONDecodeError as e:
            return {
                'TITLE': '', 'AUTHORS': '', 'AUTHORS_APA': '',
                'KEYWORDS': '', 'ABSTRACT': '', 'REFERENCES': '',
                '_ERROR': f'JSON parse error: {e}',
            }

        except Exception as e:
            error_msg = str(e)
            print(f'  Attempt {attempt}/{max_retries} failed: {error_msg}')
            if attempt < max_retries:
                wait = 2 ** attempt  # Exponential backoff: 2s, 4s, 8s
                print(f'  Waiting {wait}s before retry...')
                time.sleep(wait)
            else:
                return {
                    'TITLE': '', 'AUTHORS': '', 'AUTHORS_APA': '',
                    'KEYWORDS': '', 'ABSTRACT': '', 'REFERENCES': '',
                    '_ERROR': error_msg,
                }


print('extract_metadata_from_markdown() defined.')
print(f'Groq client ready (model: {MODEL}).')

In [ ]:
# Cell 9: Checkpoint functions — load_checkpoint() and save_checkpoint()
import json
from pathlib import Path


def load_checkpoint(path: str) -> dict:
    """Load previously processed results. Returns dict keyed by filepath string."""
    p = Path(path)
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'Checkpoint loaded: {len(data)} records found at {path}')
        return data
    print(f'No checkpoint found at {path}. Starting fresh.')
    return {}


def save_checkpoint(results: dict, path: str):
    """Persist the results dict to the checkpoint JSON file."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


print('Checkpoint functions defined.')

In [ ]:
# Cell 10: Main processing loop — process_all_articles()
import time
from tqdm.notebook import tqdm


def process_all_articles(
    manifest: list,
    client: Groq,
    model: str,
    max_chars: int,
    checkpoint_path: str,
    save_every: int = 5,
) -> dict:
    """
    Process all articles in the manifest.
    - Skips already-processed files (checkpoint resume).
    - Saves checkpoint every `save_every` newly processed files.
    - Applies 1-second rate-limiting delay between API calls.
    Returns the full results dict.
    """
    results = load_checkpoint(checkpoint_path)
    newly_processed = 0

    for item in tqdm(manifest, desc='Processing articles'):
        file_key = str(item['filepath'])

        # Skip already processed files
        if file_key in results:
            continue

        # Read markdown content
        try:
            with open(item['filepath'], 'r', encoding='utf-8') as f:
                content = f.read()
        except Exception as e:
            print(f'ERROR reading {item["filename"]}: {e}')
            content = ''

        # LLM metadata extraction
        metadata = extract_metadata_from_markdown(client, content, model, max_chars)

        # Enrich with folder-derived fields
        folder_meta = parse_folder_metadata(item['folder_name'])
        metadata.update(folder_meta)
        metadata['SOURCE_FILE'] = item['filename']
        metadata['SOURCE_FOLDER'] = item['folder_name']

        results[file_key] = metadata
        newly_processed += 1

        # Periodic checkpoint save
        if newly_processed % save_every == 0:
            save_checkpoint(results, checkpoint_path)
            print(f'  Checkpoint saved ({len(results)} total records).')

        # Rate limiting (1 second between Groq API calls)
        time.sleep(1)

    # Final checkpoint save
    save_checkpoint(results, checkpoint_path)
    print(f'\nDone! Total records in results: {len(results)}')
    return results


print('process_all_articles() defined.')

In [ ]:
# Cell 11: Execute the processing pipeline
results = process_all_articles(
    manifest=manifest,
    client=client,
    model=MODEL,
    max_chars=MAX_CONTENT_CHARS,
    checkpoint_path=CHECKPOINT_PATH,
    save_every=SAVE_EVERY,
)

In [ ]:
# Cell 12: Build DataFrame and export to CSV and Excel
import pandas as pd
from pathlib import Path


def results_to_dataframe(results: dict) -> pd.DataFrame:
    """Convert the results dict to a clean, column-ordered DataFrame."""
    records = list(results.values())
    df = pd.DataFrame(records)

    # Desired column order
    col_order = [
        'TITLE', 'AUTHORS', 'AUTHORS_APA', 'KEYWORDS', 'ABSTRACT',
        'REFERENCES', 'CONFERENCE', 'YEAR', 'SOURCE_FILE', 'SOURCE_FOLDER',
    ]
    existing_cols = [c for c in col_order if c in df.columns]
    extra_cols = [c for c in df.columns if c not in col_order]
    df = df[existing_cols + extra_cols]
    return df


df = results_to_dataframe(results)

output_base = Path(BASE_PATH)
csv_path   = output_base / 'metadados_iccrts.csv'
excel_path = output_base / 'metadados_iccrts.xlsx'

df.to_csv(csv_path, index=False, encoding='utf-8-sig')
df.to_excel(excel_path, index=False, engine='openpyxl')

print(f'DataFrame shape: {df.shape}')
print(f'CSV  saved  → {csv_path}')
print(f'Excel saved → {excel_path}')

In [ ]:
# Cell 13: Quality check and summary statistics

def generate_summary(df: pd.DataFrame):
    """Print extraction quality summary statistics."""
    print('=' * 60)
    print(f'EXTRACTION SUMMARY')
    print('=' * 60)
    print(f'Total articles processed : {len(df)}')

    print('\nArticles per conference:')
    if 'YEAR' in df.columns and 'CONFERENCE' in df.columns:
        print(df.groupby(['YEAR', 'CONFERENCE']).size().to_string())

    print('\nMissing values per field:')
    for col in ['TITLE', 'AUTHORS', 'KEYWORDS', 'ABSTRACT', 'REFERENCES']:
        if col in df.columns:
            empty = (df[col].isna() | (df[col] == '')).sum()
            pct   = empty / len(df) * 100 if len(df) > 0 else 0
            print(f'  {col:<15s}: {empty:4d} empty ({pct:.1f}%)')

    if '_ERROR' in df.columns:
        errors = df['_ERROR'].notna().sum()
        print(f'\nArticles with errors     : {errors}')
        if errors > 0:
            print('\nError breakdown:')
            print(df[df['_ERROR'].notna()][['SOURCE_FILE', '_ERROR']].to_string())
    print('=' * 60)


generate_summary(df)

In [ ]:
# Cell 14: Display sample rows for visual inspection
import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 10)

print('--- Sample: first 5 rows ---')
display(df[['TITLE', 'AUTHORS', 'KEYWORDS', 'YEAR', 'CONFERENCE', 'SOURCE_FILE']].head(5))

print('\n--- Sample: ABSTRACT column (first 3 rows) ---')
for i, row in df.head(3).iterrows():
    print(f'\n[{i}] {row.get("SOURCE_FILE", "")} ({row.get("YEAR", "")})')
    abstract = str(row.get('ABSTRACT', ''))
    print(abstract[:500] + ('...' if len(abstract) > 500 else ''))